# EVOLVE – Baseline Evaluation (YOLOv8 COCO)

This notebook evaluates a **baseline object detector** on the EVOLVE dataset.

**Baseline definition:**
- Architecture: YOLOv8s
- Weights: pretrained on COCO
- No fine-tuning on EVOLVE
- Evaluation only (zero-shot domain transfer)

The goal is to quantify the **domain gap** induced by extreme low-light and crowd conditions.

In [ ]:
import os
import yaml
import torch
import pandas as pd

from ultralytics import YOLO

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
PROJECT_ROOT = "/content/EVOLVE"  # adapt if needed
DATA_YAML = os.path.join(PROJECT_ROOT, "data", "evolve.yaml")
BASELINE_WEIGHTS = "yolov8s.pt"

assert os.path.exists(DATA_YAML), "data.yaml not found"

In [ ]:
with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

data_cfg

In [ ]:
model = YOLO(BASELINE_WEIGHTS)
model.model.eval()

print("Loaded baseline model:", BASELINE_WEIGHTS)

In [ ]:
results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    conf=0.25,
    iou=0.5,
    device=0 if torch.cuda.is_available() else "cpu",
    verbose=True
)

In [ ]:
metrics = results.metrics

print("mAP@0.5:", metrics.map50)
print("mAP@0.5:0.95:", metrics.map)
print("Precision:", metrics.precision)
print("Recall:", metrics.recall)

In [ ]:
names = results.names
stats = results.box

rows = []
for i, cls in enumerate(names):
    rows.append({
        "class": cls,
        "precision": stats.p[i],
        "recall": stats.r[i],
        "mAP@0.5": stats.ap50[i]
    })

baseline_df = pd.DataFrame(rows)
baseline_df

In [ ]:
OUT_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(OUT_DIR, exist_ok=True)

baseline_df.to_csv(os.path.join(OUT_DIR, "baseline_metrics.csv"), index=False)

## Baseline Summary

This zero-shot evaluation reveals a strong domain gap between COCO-trained detectors
and extreme low-light concert environments.

These results motivate domain-specific fine-tuning, as explored in the EVOLVE training pipeline.